In [ ]:
import os, warnings
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

warnings.filterwarnings("ignore")

In [5]:
DATA_DIR = r"D:\majorprojectdataset\ISL_CSLRT_Corpus\ISL_CSLRT_Corpus\Frames_Word_Level"  # 🔥 your dataset path

IMG_SIZE = 160
BATCH_SIZE = 16

EPOCHS = 40        # 🔥 increased
LR = 1e-5          # 🔥 lower for stability
PATIENCE = 7       # 🔥 early stopping patience

SAVE_PATH = "resnet18_model.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


In [6]:
classes = sorted([d for d in os.listdir(DATA_DIR)
                  if os.path.isdir(os.path.join(DATA_DIR, d))])

label_map = {cls: idx for idx, cls in enumerate(classes)}
idx2word = {v: k for k, v in label_map.items()}
NUM_CLASSES = len(classes)

all_paths, all_labels = [], []

for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith((".jpg",".jpeg",".png")):
            all_paths.append(os.path.join(cls_dir, f))
            all_labels.append(label_map[cls])

print("Classes:", NUM_CLASSES)
print("Total images:", len(all_paths))

Classes: 114
Total images: 11411


In [7]:
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [8]:
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    all_paths, all_labels, test_size=0.30,
    stratify=all_labels, random_state=42
)

X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50,
    stratify=y_tmp, random_state=42
)

print(f"Train={len(X_tr)} Val={len(X_val)} Test={len(X_te)}")

Train=7987 Val=1712 Test=1712


In [9]:
class ISLDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

In [10]:
train_loader = DataLoader(
    ISLDataset(X_tr, y_tr, train_tfm),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0   # 🔥 important for Jupyter/Windows
)

val_loader = DataLoader(
    ISLDataset(X_val, y_val, val_tfm),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    ISLDataset(X_te, y_te, val_tfm),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [11]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last block
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace classifier
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(DEVICE)

In [12]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

In [13]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    loss_sum, correct, total = 0, 0, 0

    for imgs, labels in tqdm(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        if train:
            optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        if train:
            loss.backward()
            optimizer.step()

        loss_sum += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)

    return loss_sum/total, correct/total

In [14]:
best_acc = 0
no_improve = 0

for ep in range(EPOCHS):
    print(f"\nEpoch {ep+1}/{EPOCHS}")

    tl, ta = run_epoch(train_loader, True)
    vl, va = run_epoch(val_loader, False)

    scheduler.step()

    print(f"Train Acc: {ta:.4f} | Val Acc: {va:.4f}")

    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), SAVE_PATH)
        print("✅ Model Saved")
        no_improve = 0
    else:
        no_improve += 1

    # Always save last model (safety)
    torch.save(model.state_dict(), "last_model.pth")

    if no_improve >= PATIENCE:
        print("⛔ Early stopping")
        break

print("\n🏆 Best Validation Accuracy:", best_acc)


Epoch 1/15


100%|████████████████████████████████████████████████████████████████████████████████| 107/107 [00:51<00:00,  2.08it/s]


Train Acc: 0.0957 | Val Acc: 0.4831
✅ Model Saved

Epoch 2/15


100%|████████████████████████████████████████████████████████████████████████████████| 107/107 [00:38<00:00,  2.81it/s]


Train Acc: 0.4052 | Val Acc: 0.6822
✅ Model Saved

Epoch 3/15


...

Train Acc: 0.8842 | Val Acc: 0.8400

Epoch 14/15


100%|████████████████████████████████████████████████████████████████████████████████| 107/107 [00:37<00:00,  2.87it/s]


Train Acc: 0.8844 | Val Acc: 0.8470
✅ Model Saved

Epoch 15/15


100%|████████████████████████████████████████████████████████████████████████████████| 107/107 [00:37<00:00,  2.86it/s]

Train Acc: 0.8878 | Val Acc: 0.8411

🏆 Best Validation Accuracy: 0.8469626168224299


In [15]:
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

correct, total = 0, 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        outputs = model(imgs)
        preds = outputs.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("✅ Test Accuracy:", correct / total)

✅ Test Accuracy: 0.8317757009345794


In [16]:
import torch

In [17]:

from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

[[14  0  0 ...  0  0  0]
 [ 0 14  0 ...  0  0  1]
 [ 0  0 15 ...  0  0  0]
 ...
 [ 0  0  0 ... 13  0  1]
 [ 0  0  0 ...  0 12  0]
 [ 0  0  0 ...  0  0 11]]
              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       1.00      0.93      0.97        15
           2       0.79      1.00      0.88        15
           3       0.94      1.00      0.97        15
           4       0.81      0.87      0.84        15
           5       0.91      0.67      0.77        15
           6       0.74      0.93      0.82        15
           7       1.00      0.80      0.89        15
           8       1.00      0.80      0.89        15
           9       0.81      0.87      0.84        15
          10       1.00      1.00      1.00        15
          11       0.92      0.80      0.86        15
          12       0.87      0.87      0.87        15
          13       1.00      0.93      0.97        15
          14       0.88      0.93